# §43 / §43.1: pre-onset rise on Nexar val, DoTA-CAP and DADA (CPU only, reads saved scores)

Rule: `core/docs/final/KIM_RUN.md` **§43** (Nexar) and **§43.1** (post-hoc: the same reads on whole DoTA-CAP (1,129) and DADA (370) for K5f, K5dfi, L0dfi, LaGoVAD). No model is loaded and nothing is trained: the tool reads the saved `nexar_val` curves (`scores/*.npz`, 225 whole videos, protocol N-B) of K5f (control), K5df (DVS `span`), K5dfi (DVS `ignore`), K5fno (normals from negative videos only) and the released LaGoVAD. **Printed only, decides nothing; K5f stays.**

| step | what | GPU | estimate |
|---|---|---|---|
| 1-2 | mount, setup; every Nexar row must have 225 saved curves, every DoTA / DADA score file must exist | no | 1 min |
| 3 | pytest on the tool - **raises if red** | no | < 1 min |
| 4 | `core.tools.pre_onset_read --plots` per set: levels + rise, zone AUC vs negative videos (Nexar), event-locked curve, §4 fixed composition + macro cost, figure | no | 1-2 min |
| 5 | **paste `REPORTS/pre_onset/pre_onset_read{,_dota,_dada}.md` to Claude** | no | - |

Reads `Thesis-Final/outputs/<row dirs>` (read-only); the report goes to `Thesis-Final/outputs/kim_fixed/REPORTS/pre_onset/`. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, the saved reads, the report
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

REPO = Path(f'{DRIVE_F}/kat-vad')
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert (REPO / 'core/tools/pre_onset_read.py').is_file(), 'this upload has no pre_onset_read -- re-upload kat-vad/'
assert '### 43.1' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), \
    'KIM_RUN.md has no §43.1 -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

ROWS = {name: constants.OUTPUT_ROOT / rel for name, rel in constants.PRE_ONSET_ROWS.items()}
REPORTS = constants.OUTPUT_ROOT / constants.PRE_ONSET_REPORT_DIR
for name, path in ROWS.items():   # every read must be the whole nexar_val (225 videos)
    n = len(list((path / 'scores').glob('*.npz')))
    assert n == 225, f'{name}: {n} saved curves in {path}/scores, expected 225 (nexar_val)'
    print(f'{name:8s} {path}')
ZS_ROWS = {s: {n: constants.OUTPUT_ROOT / rel for n, rel in rows.items()}
           for s, rows in constants.PRE_ONSET_ZS_ROWS.items()}
for set_name, rows in ZS_ROWS.items():
    for name, path in rows.items():
        assert path.is_file(), f'{set_name}/{name}: no clip_scores.npz at {path}'
        print(f'{set_name:5s} {name:8s} {path}')
print('report ->', REPORTS)


## 3. Tests on the tool (CPU) - **raises if anything fails: stop and paste the output**

In [ ]:
ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')


def run(cmd):
    '''Stream a child's output; raise with its tail on failure.'''
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'])
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_pre_onset_read.py'])


## 4. The read (`core.tools.pre_onset_read`): refuses if the rows' videos or labels differ

In [ ]:
from IPython.display import Image, display

stem = constants.PRE_ONSET_REPORT_STEM
jobs = [('nexar', ROWS, f'{stem}.md', 'pre_onset_curves.png')]
jobs += [(s, rows, f'{stem}_{s}.md', f'pre_onset_curves_{s}.png') for s, rows in ZS_ROWS.items()]
for set_name, rows, report, figure in jobs:
    args = [sys.executable, '-m', 'core.tools.pre_onset_read', '--set', set_name,
            '--control', constants.PRE_ONSET_CONTROL, '--out-dir', REPORTS, '--plots']
    for name, path in rows.items():
        args += ['--row', f'{name}={path}']
    run(args)
    print((REPORTS / report).read_text())
    display(Image(filename=str(REPORTS / figure)))
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f'Drive flushed -- paste {REPORTS}/{stem}{{,_dota,_dada}}.md to Claude')
